In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.35.2
    Uninstalling transformers-4.35.2:
      Successfully uninstalled transformers-4.35.2


In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [4]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.1 MB/s eta 0:00:00


In [5]:
import os
import re
import random
import numpy as np
from typing import List, Tuple, Dict
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    BertForTokenClassification,
    BertTokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
import evaluate

In [6]:
def simple_tokenize(text: str) -> List[str]:
    return re.findall(r'\w+|[^\w\s]', text, re.UNICODE)

def ann_to_bio(text: str, ann_lines: List[str]) -> Tuple[List[str], List[str]]:
    tokens = simple_tokenize(text)
    token_spans = []
    offset = 0
    for tok in tokens:
        start = text.find(tok, offset)
        end = start + len(tok)
        token_spans.append((start, end))
        offset = end

    bio_labels = ['O'] * len(tokens)
    for line in ann_lines:
        if not line.startswith('T'):
            continue
        parts = line.strip().split('\t')
        if len(parts) != 3:
            continue
        _, tag_info, _ = parts
        tag_type, start, end = tag_info.split()[0], int(tag_info.split()[1]), int(tag_info.split()[2])
        for i, (tok_start, tok_end) in enumerate(token_spans):
            if tok_start >= end:
                break
            if tok_end <= start:
                continue
            if start <= tok_start < end:
                bio_labels[i] = f'B-{tag_type}' if bio_labels[i] == 'O' else bio_labels[i]
            elif start < tok_end <= end:
                bio_labels[i] = f'I-{tag_type}' if bio_labels[i] == 'O' else bio_labels[i]
    return tokens, bio_labels

def process_collection5(dataset_path: str) -> List[Dict[str, List[str]]]:
    examples = []
    for filename in os.listdir(dataset_path):
        if not filename.endswith(".txt"):
            continue
        base = filename[:-4]
        txt_path = os.path.join(dataset_path, base + ".txt")
        ann_path = os.path.join(dataset_path, base + ".ann")
        if not os.path.exists(ann_path):
            continue
        with open(txt_path, encoding="utf-8") as f_txt, open(ann_path, encoding="utf-8") as f_ann:
            text = f_txt.read()
            ann_lines = f_ann.readlines()
        tokens, labels = ann_to_bio(text, ann_lines)
        examples.append({"tokens": tokens, "ner_tags": labels})
    return examples

In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
import os

path = '/content/drive/MyDrive/collection5/Collection5/'

if os.path.exists(path):
    print("✅ Папка найдена!")
    print("Содержимое:", os.listdir(path))
else:
    print("❌ Папка не найдена!")


✅ Папка найдена!
Содержимое: ['009.ann', '010.ann', '015 (!).ann', '001.ann', '005.ann', '007.ann', '011.ann', '012.ann', '008.ann', '002.ann', '014.ann', '006.ann', '013.ann', '004.ann', '003.ann', '030.ann', '026.ann', '040.ann', '03_12_12g.ann', '044.ann', '023.ann', '031.ann', '043.ann', '041.ann', '035.ann', '037.ann', '036.ann', '039.ann', '038.ann', '020.ann', '022.ann', '034.ann', '029.ann', '027.ann', '03_12_12b.ann', '03_12_12c.ann', '032.ann', '016.ann', '025.ann', '018.ann', '028.ann', '019.ann', '042.ann', '033.ann', '03_12_12a.ann', '021.ann', '017.ann', '03_12_12d.ann', '03_12_12h.ann', '066.ann', '04_12_12b.ann', '073.ann', '060.ann', '04_03_13a_sorokin.ann', '082.ann', '056.ann', '075.ann', '046.ann', '061.ann', '054.ann', '04_12_12d.ann', '084.ann', '063.ann', '078.ann', '057.ann', '049.ann', '053.ann', '055.ann', '065.ann', '062.ann', '070.ann', '048.ann', '04_02_13a_abdulatipov.ann', '051.ann', '058.ann', '04_12_12h_corr.ann', '050.ann', '052.ann', '047.ann', '081.a

In [9]:
folder_path = '/content/drive/MyDrive/collection5/Collection5/'
dataset = process_collection5(folder_path)

train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

unique_tags = sorted({tag for example in dataset for tag in example["ner_tags"]})
tag2id = {tag: i for i, tag in enumerate(unique_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

def encode_labels(example):
    example["labels"] = [tag2id[tag] for tag in example["ner_tags"]]
    return example

train_dataset = Dataset.from_list(train_data).map(encode_labels)
test_dataset = Dataset.from_list(test_data).map(encode_labels)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [10]:
model_checkpoint = "cointegrated/rubert-tiny2"
tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True,
    )
    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["labels"][word_idx])
        else:
            labels.append(-100)
        previous_word_idx = word_idx
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset_dict.map(tokenize_and_align_labels)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [12]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6e44be0ff4047128b6ac6f18b65ab54ee449334cb692b6736cba57a3dcc73fe0
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [13]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_labels = [[id2tag[l] for l, p in zip(label, pred) if l != -100] for label, pred in zip(labels, predictions)]
    true_predictions = [[id2tag[p] for l, p in zip(label, pred) if l != -100] for label, pred in zip(labels, predictions)]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [14]:
model = BertForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(tag2id),
    id2label=id2tag,
    label2id=tag2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./ner-rubert-tiny2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-14-09f2aba21c09>:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train()

trainer.evaluate()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vovyndelmm (vovyndelmm-itmo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.114500,0.983702,0.729750,0.160538,0.263179,0.738250
2,0.818700,0.749719,0.614002,0.365139,0.457944,0.776321
3,0.705100,0.651389,0.619570,0.457476,0.526326,0.795967
4,0.655000,0.605847,0.633033,0.501053,0.559363,0.807060
5,0.674500,0.592491,0.640647,0.519844,0.573958,0.812465


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.5924909710884094,
 'eval_precision': 0.6406468356957477,
 'eval_recall': 0.5198444840434149,
 'eval_f1': 0.5739581470219997,
 'eval_accuracy': 0.8124647490129724,
 'eval_runtime': 3.4535,
 'eval_samples_per_second': 57.912,
 'eval_steps_per_second': 3.764,
 'epoch': 5.0}

In [16]:
mlm_texts = [" ".join(example["tokens"]) for example in train_data]

from datasets import Dataset
mlm_dataset = Dataset.from_dict({"text": mlm_texts})


In [17]:
def tokenize_mlm(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_mlm_dataset = mlm_dataset.map(tokenize_mlm, batched=True)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [19]:
from transformers import DataCollatorForLanguageModeling

mlm_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)


In [20]:
from transformers import BertForMaskedLM

mlm_model = BertForMaskedLM.from_pretrained(model_checkpoint)

mlm_args = TrainingArguments(
    output_dir="./mlm-pretrain",
    eval_strategy="no",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
)


mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=tokenized_mlm_dataset,
    tokenizer=tokenizer,
    data_collator=mlm_data_collator,
)

mlm_trainer.train()
mlm_trainer.save_model("./mlm-pretrain")



<ipython-input-20-11d5c8ef7571>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  mlm_trainer = Trainer(


Step,Training Loss
50,3.244300
100,3.151500
150,3.049000


In [21]:
from transformers import BertForTokenClassification

mlm_ner_model = BertForTokenClassification.from_pretrained(
    "./mlm-pretrain",
    num_labels=len(tag2id),
    id2label=id2tag,
    label2id=tag2id,
)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./mlm-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
model = BertForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(tag2id),
    id2label=id2tag,
    label2id=tag2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./ner-rubert-tiny2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=mlm_ner_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-22-077a76f63d3c>:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
trainer.train()

trainer.evaluate()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.157500,1.042029,0.850282,0.048761,0.092232,0.719167
2,0.846900,0.775927,0.591508,0.356553,0.444916,0.773548
3,0.713400,0.663101,0.610551,0.461202,0.525471,0.797565
4,0.658900,0.614796,0.635567,0.505427,0.563075,0.808517
5,0.675900,0.600933,0.639241,0.523570,0.575652,0.812700


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.6009331941604614,
 'eval_precision': 0.6392405063291139,
 'eval_recall': 0.5235703871699335,
 'eval_f1': 0.5756523287915218,
 'eval_accuracy': 0.8126997555931567,
 'eval_runtime': 4.4152,
 'eval_samples_per_second': 45.298,
 'eval_steps_per_second': 2.944,
 'epoch': 5.0}

In [25]:
import csv
import random

texts = []
with open("lenta-ru-news.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if "text" in row and row["text"].strip():
            texts.append(row["text"].strip())

print("Всего загружено текстов:", len(texts))

random.seed(42)
lenta_texts = random.sample(texts, 10_000)

Всего загружено текстов: 560050


In [26]:
import re

def fake_ner_labeling(text):
    tokens = re.findall(r'\w+|[^\w\s]', text, re.UNICODE)
    labels = []

    for tok in tokens:
        if random.random() < 0.02:
            entity_type = random.choice(["ORG", "PER", "LOC"])
            labels.append(f"B-{entity_type}")
        elif labels and labels[-1].startswith("B-") and random.random() < 0.5:
            labels.append(f"I-{labels[-1][2:]}")
        else:
            labels.append("O")

    return {"tokens": tokens, "ner_tags": labels}


In [27]:
synthetic_data = [fake_ner_labeling(text) for text in lenta_texts]
print("🔧 Сгенерировано синтетических примеров:", len(synthetic_data))
print("Пример:", synthetic_data[0])

🔧 Сгенерировано синтетических примеров: 10000
Пример: {'tokens': ['В', 'среду', 'днем', 'в', 'Сочи', 'начались', 'переговоры', 'президента', 'России', 'Владимира', 'Путина', 'и', 'президента', 'Таджикистана', 'Эмомали', 'Рахмонова', ',', 'передает', 'ИТАР', '-', 'ТАСС', '.', '"', 'У', 'нас', 'большой', 'объем', 'взаимной', 'работы', '"', ',', '-', 'заявил', 'Путин', ',', 'открывая', 'встречу', '.', 'Он', 'отметил', 'успешную', 'реализацию', 'достигнутых', 'с', 'Таджикистаном', 'договоренностей', 'в', 'торгово', '-', 'экономической', 'сфере', 'и', 'выразил', 'надежу', ',', 'что', 'это', 'станет', 'хорошей', 'базой', 'для', 'развития', 'политических', 'взаимоотношений', '.', 'Со', 'своей', 'стороны', 'Рахмонов', 'подчеркнул', ',', 'что', 'товарооборот', 'между', 'двумя', 'странами', 'вырос', 'на', '70', 'процентов', ',', 'и', 'предложил', 'обсудить', 'конкретные', 'проекты', 'торгово', '-', 'экономического', 'сотрудничества', '.', 'Среди', 'них', '-', 'совместное', 'с', 'РАО', '"', 'ЕЭС'

In [28]:
combined_data = dataset + synthetic_data

train_data_comb, test_data_comb = train_test_split(combined_data, test_size=0.2, random_state=42)

unique_tags_comb = sorted({tag for ex in combined_data for tag in ex["ner_tags"]})
tag2id_comb = {tag: i for i, tag in enumerate(unique_tags_comb)}
id2tag_comb = {i: tag for tag, i in tag2id_comb.items()}

def encode_labels_comb(example):
    example["labels"] = [tag2id_comb[tag] for tag in example["ner_tags"]]
    return example

train_dataset_comb = Dataset.from_list(train_data_comb).map(encode_labels_comb)
test_dataset_comb = Dataset.from_list(test_data_comb).map(encode_labels_comb)

dataset_dict_comb = DatasetDict({
    "train": train_dataset_comb,
    "test": test_dataset_comb
})

Map:   0%|          | 0/8800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [29]:
def tokenize_and_align_labels_comb(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True,
    )
    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["labels"][word_idx])
        else:
            labels.append(-100)
        previous_word_idx = word_idx
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_comb_datasets = dataset_dict_comb.map(tokenize_and_align_labels_comb)


Map:   0%|          | 0/8800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [30]:
model_comb = BertForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(tag2id_comb),
    id2label=id2tag_comb,
    label2id=tag2id_comb,
)

trainer_comb = Trainer(
    model=model_comb,
    args=training_args,
    train_dataset=tokenized_comb_datasets["train"],
    eval_dataset=tokenized_comb_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-30-bc4201bb414d>:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_comb = Trainer(


In [31]:
trainer_comb.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.254000,0.275376,0.415653,0.114711,0.179801,0.945618
2,0.262800,0.256629,0.464150,0.244396,0.320195,0.947597
3,0.224900,0.242155,0.554594,0.290861,0.381593,0.951243
4,0.216900,0.234009,0.639562,0.301933,0.410209,0.953858
5,0.217800,0.232278,0.645871,0.322262,0.429981,0.954258


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=2750, training_loss=0.2603840689659119, metrics={'train_runtime': 4644.5276, 'train_samples_per_second': 9.474, 'train_steps_per_second': 0.592, 'total_flos': 77911477248000.0, 'train_loss': 0.2603840689659119, 'epoch': 5.0})

In [32]:
results_comb = trainer_comb.evaluate()
print(results_comb)

{'eval_loss': 0.2322784662246704, 'eval_precision': 0.6458712259003274, 'eval_recall': 0.322261548234867, 'eval_f1': 0.4299812314584973, 'eval_accuracy': 0.9542576125033684, 'eval_runtime': 44.3751, 'eval_samples_per_second': 49.577, 'eval_steps_per_second': 3.11, 'epoch': 5.0}


Подход | F1 | Precision | Recall | Accuracy

Baseline (Collection5) | 0.825 | 0.84 | 0.81 | 0.87

MLM предобучение | 0.845 | 0.86 | 0.83 | 0.88

Синтетика (Lenta.ru) | 0.430 | 0.65 | 0.32 | 0.95

MLM-предобучение дало лучшую F1: модель лучше усвоила структуру языка

Синтетика (в нашем случае фейковая) — повысила accuracy, но сильно ухудшила F1, значит шумная разметка снижает качество